In [13]:
import subprocess
import time
import os

# 1. Установка системных пакетов и Python-зависимостей
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!pip install -q "pandas==2.2.2" "numpy>=2.0,<2.6" "httpx==0.28.1" "fsspec==2025.3.0" "huggingface-hub>=1.2.0,<2.0" "datasets==5.0.0" "python-dotenv==1.2.2" "langchain-core==1.4.8" "langchain-qdrant==1.1.0" "langchain-text-splitters==1.1.2" "langchain-ollama==1.1.0" "ragas==0.4.3" "sentence-transformers==5.6.0" "transformers==5.13.0"
!pip install -q langchain_google_vertexai langchain_google_genai

# 2. Установка Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Безопасный запуск Ollama в фоне через subprocess (обходим ошибку Kaggle)
print("Запуск Ollama сервера на GPU 0...")
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_KEEP_ALIVE"] = "30m"
env["OLLAMA_NUM_PARALLEL"] = "4"

with open("ollama.log", "w") as log_file:
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

# Ждем 5 секунд, чтобы сервер успел подняться
time.sleep(5)


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,416 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe am

In [15]:
# Проверяем статус и качаем модель
!curl http://localhost:11434

Ollama is running

In [ ]:

!ollama pull qwen2.5-coder:7b

In [16]:
import os
import sys
import torch
from types import ModuleType
from dotenv import load_dotenv
import langchain_google_vertexai

# 1. Автоматически находим, где именно лежит .env в директории input
ENV_PATH = None
DATASET_DIR = None

for root, dirs, files in os.walk("/kaggle/input"):
    if ".env" in files:
        ENV_PATH = os.path.join(root, ".env")
        DATASET_DIR = root
        break

# Если автопоиск почему-то не отработал, ставим прямой фолбек
if not ENV_PATH:
    DATASET_DIR = "/kaggle/input/uesp-wiki-dataset"  # или "/kaggle/input/UESP_Wiki_Dataset"
    ENV_PATH = f"{DATASET_DIR}/.env"

# 2. Загружаем ключи
if os.path.exists(ENV_PATH):
    load_dotenv(dotenv_path=ENV_PATH, override=True)
    print(f"✓ Файл .env успешно найден по пути: {ENV_PATH}")
else:
    print(f"✕ КРИТИЧЕСКАЯ ОШИБКА: Не удалось найти .env по пути {ENV_PATH}")

# Настраиваем пути к датасетам на основе найденной папки
INPUT_FILE_PATH = os.path.join(DATASET_DIR, "Oblivion_Cleaned_fixed.jsonl")
GOLDEN_SET_PATH = os.path.join(DATASET_DIR, "golden_set_by_claude.json")
OUTPUT_PATH = "/kaggle/working/reranker_eval_results.jsonl" 

# Проверяем, что всё прочиталось нормально
print("QDRANT_URL =", repr(os.getenv("QDRANT_URL")))
print("QDRANT_API_KEY =", repr(os.getenv("QDRANT_API_KEY")[:12] + "..." if os.getenv("QDRANT_API_KEY") else None))

# 3. Паркуем тяжелый PyTorch на вторую карту cuda:1
if torch.cuda.device_count() > 1:
    MODEL_DEVICE = "cuda:1"
    print(f"\n✓ Эмбеддинги и реранкер успешно припаркованы на {MODEL_DEVICE}")
else:
    MODEL_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"\n! Доступно GPU: {torch.cuda.device_count()}. Выбран: {MODEL_DEVICE}")

# Симуляция старых модулей для совместимости
if "langchain_community.chat_models.vertexai" not in sys.modules:
    mock_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
    mock_chat_vertex.ChatVertexAI = langchain_google_vertexai.ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = mock_chat_vertex

if "langchain_community.llms" not in sys.modules:
    mock_llms = ModuleType("langchain_community.llms")
    mock_llms.VertexAI = langchain_google_vertexai.VertexAI
    sys.modules["langchain_community.llms"] = mock_llms

✓ Файл .env успешно найден по пути: /kaggle/input/datasets/okharitonov/uesp-wiki-dataset/.env
QDRANT_URL = 'https://b50da76c-c016-433d-b4dd-93f354851313.europe-west3-0.gcp.cloud.qdrant.io'
QDRANT_API_KEY = 'eyJhbGciOiJI...'

✓ Эмбеддинги и реранкер успешно припаркованы на cuda:1


In [17]:
import gc
import itertools
import json
from datetime import datetime, timezone
from typing import Sequence

import pandas as pd
from datasets import Dataset
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_ollama import ChatOllama
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from ragas.run_config import RunConfig
from sentence_transformers import CrossEncoder, SentenceTransformer

COLLECTION_NAME = "oblivion_lore"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 124
RETRIEVE_K = 10
BATCH_SIZE = 400
EVAL_SAMPLE_SIZE = 20
RERANK_TOP_N_VALUES = [5, 7]

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
CROSS_ENCODER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str, device: str):
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

class RerankingRetriever:
    def __init__(self, vector_store: QdrantVectorStore, cross_encoder: CrossEncoder, top_n: int):
        self.vector_store = vector_store
        self.cross_encoder = cross_encoder
        self.top_n = top_n

    def retrieve(self, query: str) -> list[Document]:
        documents = self.vector_store.similarity_search(query, k=RETRIEVE_K)
        if not documents: return []
        pairs = [(query, doc.page_content) for doc in documents]
        scores = self.cross_encoder.predict(pairs)
        scored_documents = sorted(zip(documents, scores, strict=True), key=lambda x: float(x[1]), reverse=True)
        return [doc for doc, _ in scored_documents[: self.top_n]]

# Инициализация моделей на GPU 1
embeddings = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL_NAME, device=MODEL_DEVICE)
cross_encoder = CrossEncoder(model_name=CROSS_ENCODER_MODEL_NAME, device=MODEL_DEVICE)

eval_llm = ChatOllama(model="qwen2.5-coder:7b", temperature=0.0)
judge_llm = ChatOllama(model="qwen2.5-coder:7b", temperature=0.0, timeout=3600, num_ctx=4096)

def load_golden_set(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл голден-сета не найден по пути: {path}")
        
    if path.endswith(".json"):
        print(f"✓ Загрузка датасета в формате JSON из {path}")
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        # Превращаем JSON-массив в DataFrame (поля user_input и reference подхватятся автоматически)
        return pd.DataFrame(data)
    else:
        print(f"✓ Загрузка датасета в формате CSV из {path}")
        return pd.read_csv(path)

def load_raw_documents(input_file_path: str) -> list[Document]:
    raw_documents = []
    with open(input_file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            metadata = data["metadata"]
            for section_title, section_text in data["clean_text"].items():
                chunk_metadata = metadata.copy()
                chunk_metadata["section_title"] = section_title
                raw_documents.append(Document(page_content=section_text, metadata=chunk_metadata))
    return raw_documents

def upload_documents_to_qdrant(documents: list[Document]) -> QdrantVectorStore:
    print(f"Создание коллекции {COLLECTION_NAME} в Qdrant Cloud...")
    vector_store = QdrantVectorStore.from_documents(
        documents=[documents[0]], embedding=embeddings, url=QDRANT_URL, api_key=QDRANT_API_KEY, collection_name=COLLECTION_NAME, force_recreate=True
    )
    for i in range(1, len(documents), BATCH_SIZE):
        batch = documents[i : i + BATCH_SIZE]
        vector_store.add_documents(batch)
    return vector_store

def build_vector_store() -> QdrantVectorStore:
    return QdrantVectorStore.from_existing_collection(
        embedding=embeddings, collection_name=COLLECTION_NAME, url=QDRANT_URL, api_key=QDRANT_API_KEY
    )

def create_reranker_retriever(vector_store: QdrantVectorStore, top_n: int) -> RerankingRetriever:
    return RerankingRetriever(vector_store=vector_store, cross_encoder=cross_encoder, top_n=top_n)

def format_context(documents: Sequence[Document]) -> str:
    if not documents: return "Контекст не найден."
    return "\n\n".join(f"Фрагмент {idx}:\n{doc.page_content}" for idx, doc in enumerate(documents, start=1))

def get_message_text(message) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str): return content
    if isinstance(content, list): return "\n".join(p.get("text", str(p)) if isinstance(p, dict) else str(p) for p in content)
    return str(content)

def generate_answer(question: str, documents: Sequence[Document]) -> str:
    system_prompt = f"Ты эксперт по вселенной The Elder Scrolls IV: Oblivion. Используй только контекст:\n\n{format_context(documents)}"
    return get_message_text(eval_llm.invoke([("system", system_prompt), ("human", question)]))

def generate_rag_answers(retriever: RerankingRetriever, golden_set_df: pd.DataFrame) -> pd.DataFrame:
    llm_answers = []
    total = len(golden_set_df)
    
    print(f"Шаг 1: Сбор контекстов для {total} вопросов...")
    questions = golden_set_df["user_input"].tolist()
    references = golden_set_df["reference"].tolist()
    
    # Контексты собираем быстро (на GPU 1)
    all_docs = [retriever.retrieve(q) for q in questions]
    all_contexts = [[doc.page_content for doc in docs] for docs in all_docs]
    
    print("Шаг 2: Параллельная генерация ответов через LLM...")
    prompts = []
    for q, docs in zip(questions, all_docs, strict=True):
        system_prompt = (
            "Ты эксперт по вселенной The Elder Scrolls IV: Oblivion. "
            "Используй только приведенные фрагменты лора, чтобы ответить на вопрос. "
            "Если ответа нет в контексте, честно скажи, что не знаешь. Не выдумывай информацию.\n\n"
            f"Контекст:\n{format_context(docs)}"
        )
        prompts.append([("system", system_prompt), ("human", q)])
    
    # Отправляем батчем в 4 потока
    responses = eval_llm.batch(prompts, config={"max_concurrency": 4})
    
    for q, resp, ctx, ref in zip(questions, responses, all_contexts, references, strict=True):
        llm_answers.append({
            "user_input": q,
            "response": get_message_text(resp),
            "retrieved_contexts": ctx,
            "reference": ref
        })
        
    print("Генерация завершена.")
    return pd.DataFrame(llm_answers)

def evaluate_rag_metrics(results: pd.DataFrame) -> dict:
    dataset = Dataset.from_dict({
        "user_input": results["user_input"].tolist(),
        "response": results["response"].tolist(),
        "retrieved_contexts": results["retrieved_contexts"].tolist(),
        "reference": results["reference"].tolist(),
    })
    metrics_results = evaluate(
        dataset=dataset, metrics=[context_precision, context_recall], llm=judge_llm, embeddings=embeddings,
        run_config=RunConfig(timeout=3600, max_workers=4, max_retries=3), raise_exceptions=True
    )
    df = metrics_results.to_pandas()
    return {"context_precision": float(df["context_precision"].mean()), "context_recall": float(df["context_recall"].mean())}

def save_experiment_result(params: dict, metrics: dict) -> None:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    record = {"timestamp": datetime.now(timezone.utc).isoformat(), "params": params, "metrics": metrics}
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def rag_tuning_pipeline() -> None:
    vector_store = build_vector_store()

    # Используем новую функцию вместо pd.read_csv
    full_golden_set_df = load_golden_set(GOLDEN_SET_PATH)
    
    # Безопасное сэмплирование: если в датасете вопросов меньше, чем EVAL_SAMPLE_SIZE, берем все
    sample_size = min(EVAL_SAMPLE_SIZE, len(full_golden_set_df))
    print(f"Размер выборки для оценки: {sample_size} вопросов (всего в файле: {len(full_golden_set_df)})")
    
    golden_set_df = full_golden_set_df.sample(n=sample_size, random_state=42)

    experiments = [{"chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP, "retrieve_k": RETRIEVE_K, "top_n": val} for val in RERANK_TOP_N_VALUES]
    
    for idx, params in enumerate(experiments):
        print(f"\n--- Эксперимент {idx + 1}/{len(experiments)} ---")
        print(f"Параметры: {params}")
        
        retriever = create_reranker_retriever(vector_store, top_n=params["top_n"])
        results_df = generate_rag_answers(retriever, golden_set_df)
        
        print(f"[{idx + 1}/{len(experiments)}] Запуск оценки Ragas...")
        metrics = evaluate_rag_metrics(results_df)
        
        print(f"Метрики эксперимента: {metrics}")
        save_experiment_result(params, metrics)
        
        del retriever, results_df
        gc.collect()

if __name__ == "__main__":
    rag_tuning_pipeline()

/tmp/ipykernel_58/2886311405.py:15: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
/tmp/ipykernel_58/2886311405.py:15: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✓ Загрузка датасета в формате JSON из /kaggle/input/datasets/okharitonov/uesp-wiki-dataset/golden_set_by_claude.json
Размер выборки для оценки: 20 вопросов (всего в файле: 40)

--- Эксперимент 1/2 ---
Параметры: {'chunk_size': 1024, 'chunk_overlap': 124, 'retrieve_k': 10, 'top_n': 5}
Шаг 1: Сбор контекстов для 20 вопросов...
Шаг 2: Параллельная генерация ответов через LLM...
Генерация завершена.
[1/2] Запуск оценки Ragas...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Метрики эксперимента: {'context_precision': 0.6249999999483333, 'context_recall': 0.8916666666666668}

--- Эксперимент 2/2 ---
Параметры: {'chunk_size': 1024, 'chunk_overlap': 124, 'retrieve_k': 10, 'top_n': 7}
Шаг 1: Сбор контекстов для 20 вопросов...
Шаг 2: Параллельная генерация ответов через LLM...
Генерация завершена.
[2/2] Запуск оценки Ragas...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Метрики эксперимента: {'context_precision': 0.5402777777314351, 'context_recall': 0.8}
